In [1]:
from dotenv import load_dotenv
import os
import boto3
import awswrangler as wr
import pandas as pd

load_dotenv()

S3_OUTPUT = "s3://pharma-bi-raw/athena-results/"
DATABASE  = "pharma_bi_db"

session = boto3.Session(
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    region_name=os.getenv("AWS_DEFAULT_REGION")
)

def run_query(sql: str) -> pd.DataFrame:
    """Run an Athena SQL query and return a pandas DataFrame."""
    return wr.athena.read_sql_query(
        sql=sql,
        database=DATABASE,
        s3_output=S3_OUTPUT,
        boto3_session=session
    )

# Sanity check — run this first in every notebook
df_check = run_query("SELECT COUNT(*) AS n FROM fact_sales")
print(f"Connection OK — {df_check['n'].iloc[0]:,} rows in FactSales")

Connection OK — 62,139 rows in FactSales


In [2]:
query = """
SELECT 'fact_sales'   AS table_name, COUNT(*) AS row_count FROM fact_sales   UNION ALL
SELECT 'dim_date',                   COUNT(*)               FROM dim_date     UNION ALL
SELECT 'dim_pharmacy',               COUNT(*)               FROM dim_pharmacy UNION ALL
SELECT 'dim_product',                COUNT(*)               FROM dim_product
"""

df_counts = run_query(query)

# Expected counts for validation
EXPECTED = {
    'fact_sales':   62139,
    'dim_date':     731,
    'dim_pharmacy': 120,
    'dim_product':  220
}

df_counts['expected'] = df_counts['table_name'].map(EXPECTED)
df_counts['match']    = df_counts['row_count'] == df_counts['expected']

print(df_counts.to_string(index=False))
print()

if df_counts['match'].all():
    print("All row counts match expected values.")
else:
    print("WARNING: Row count mismatch detected. Check S3 upload.")

  table_name  row_count  expected  match
 dim_product        220       220   True
    dim_date        731       731   True
dim_pharmacy        120       120   True
  fact_sales      62139     62139   True

All row counts match expected values.


In [4]:
# Schema review using information_schema (Athena compatible)
tables = ['fact_sales', 'dim_date', 'dim_pharmacy', 'dim_product']

for table in tables:
    query = f"""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = '{DATABASE}'
    AND table_name = '{table}'
    ORDER BY ordinal_position
    """
    df_schema = run_query(query)
    print(f"\n── {table} ──")
    print(df_schema.to_string(index=False))


── fact_sales ──
column_name data_type
    salesid   varchar
    datekey    bigint
 pharmacyid   varchar
  productid   varchar
  unitssold    bigint
 revenueeur    double
    costeur    double
  margineur    double
  promoflag   varchar

── dim_date ──
column_name data_type
    datekey    bigint
       date   varchar
       year    bigint
    quarter    bigint
monthnumber    bigint
  monthname   varchar
  yearmonth   varchar

── dim_pharmacy ──
  column_name data_type
   pharmacyid   varchar
 pharmacyname   varchar
      country   varchar
       region   varchar
         city   varchar
 pharmacytype   varchar
     opendate   varchar
storesizeband   varchar
     latitude    double
    longitude    double

── dim_product ──
     column_name data_type
       productid   varchar
     productname   varchar
        category   varchar
           brand   varchar
       isgeneric   varchar
        packsize   varchar
    listpriceeur    double
 standardcosteur    double
      launchdate   varch

In [5]:
query_fs_nulls = """
SELECT
    COUNT(*) - COUNT(salesid)     AS salesid_nulls,
    COUNT(*) - COUNT(datekey)     AS datekey_nulls,
    COUNT(*) - COUNT(pharmacyid)  AS pharmacyid_nulls,
    COUNT(*) - COUNT(productid)   AS productid_nulls,
    COUNT(*) - COUNT(unitssold)   AS unitssold_nulls,
    COUNT(*) - COUNT(revenueeur)  AS revenueeur_nulls,
    COUNT(*) - COUNT(costeur)     AS costeur_nulls,
    COUNT(*) - COUNT(margineur)   AS margineur_nulls,
    COUNT(*) - COUNT(promoflag)   AS promoflag_nulls
FROM fact_sales
"""

print("── FactSales null counts ──")
df_fs_nulls = run_query(query_fs_nulls)
print(df_fs_nulls.T.rename(columns={0: 'null_count'}).to_string())

── FactSales null counts ──
                  null_count
salesid_nulls              0
datekey_nulls              0
pharmacyid_nulls           0
productid_nulls            0
unitssold_nulls            0
revenueeur_nulls           0
costeur_nulls              0
margineur_nulls            0
promoflag_nulls            0


In [6]:
query_dp_nulls = """
SELECT
    COUNT(*) - COUNT(pharmacyid)    AS pharmacyid_nulls,
    COUNT(*) - COUNT(pharmacyname)  AS pharmacyname_nulls,
    COUNT(*) - COUNT(country)       AS country_nulls,
    COUNT(*) - COUNT(region)        AS region_nulls,
    COUNT(*) - COUNT(city)          AS city_nulls,
    COUNT(*) - COUNT(pharmacytype)  AS pharmacytype_nulls,
    COUNT(*) - COUNT(storesizeband) AS storesizeband_nulls,
    COUNT(*) - COUNT(latitude)      AS latitude_nulls,
    COUNT(*) - COUNT(longitude)     AS longitude_nulls
FROM dim_pharmacy
"""

print("── DimPharmacy null counts ──")
df_dp_nulls = run_query(query_dp_nulls)
print(df_dp_nulls.T.rename(columns={0: 'null_count'}).to_string())

── DimPharmacy null counts ──
                     null_count
pharmacyid_nulls              0
pharmacyname_nulls            0
country_nulls                 0
region_nulls                  0
city_nulls                    0
pharmacytype_nulls            0
storesizeband_nulls           0
latitude_nulls                0
longitude_nulls               0


In [7]:
# Note: DiscontinuedDate is expected to be null for active products (185 nulls expected)
query_dpr_nulls = """
SELECT
    COUNT(*) - COUNT(productid)        AS productid_nulls,
    COUNT(*) - COUNT(productname)      AS productname_nulls,
    COUNT(*) - COUNT(category)         AS category_nulls,
    COUNT(*) - COUNT(brand)            AS brand_nulls,
    COUNT(*) - COUNT(isgeneric)        AS isgeneric_nulls,
    COUNT(*) - COUNT(listpriceeur)     AS listpriceeur_nulls,
    COUNT(*) - COUNT(standardcosteur)  AS standardcosteur_nulls,
    COUNT(*) - COUNT(isdiscontinued)   AS isdiscontinued_nulls,
    COUNT(*) - COUNT(discontinueddate) AS discontinueddate_nulls
FROM dim_product
"""

print("── DimProduct null counts ──")
df_dpr_nulls = run_query(query_dpr_nulls)
print(df_dpr_nulls.T.rename(columns={0: 'null_count'}).to_string())
print()
print("Note: discontinueddate_nulls = 185 is expected (nulls = active products).")

── DimProduct null counts ──
                        null_count
productid_nulls                  0
productname_nulls                0
category_nulls                   0
brand_nulls                      0
isgeneric_nulls                  0
listpriceeur_nulls               0
standardcosteur_nulls            0
isdiscontinued_nulls             0
discontinueddate_nulls           0

Note: discontinueddate_nulls = 185 is expected (nulls = active products).


In [8]:
query_dupes = """
SELECT
    'fact_sales'   AS table_name,
    'salesid'      AS primary_key,
    COUNT(*)       AS total_rows,
    COUNT(DISTINCT salesid) AS unique_keys,
    COUNT(*) - COUNT(DISTINCT salesid) AS duplicates
FROM fact_sales

UNION ALL

SELECT
    'dim_date', 'datekey',
    COUNT(*), COUNT(DISTINCT datekey),
    COUNT(*) - COUNT(DISTINCT datekey)
FROM dim_date

UNION ALL

SELECT
    'dim_pharmacy', 'pharmacyid',
    COUNT(*), COUNT(DISTINCT pharmacyid),
    COUNT(*) - COUNT(DISTINCT pharmacyid)
FROM dim_pharmacy

UNION ALL

SELECT
    'dim_product', 'productid',
    COUNT(*), COUNT(DISTINCT productid),
    COUNT(*) - COUNT(DISTINCT productid)
FROM dim_product
"""

df_dupes = run_query(query_dupes)
print(df_dupes.to_string(index=False))
print()

if (df_dupes['duplicates'] == 0).all():
    print("All primary keys are unique. No duplicates found.")
else:
    print("WARNING: Duplicate primary keys detected. Investigate before analysis.")

  table_name primary_key  total_rows  unique_keys  duplicates
  fact_sales     salesid       62139        62139           0
 dim_product   productid         220          220           0
dim_pharmacy  pharmacyid         120          120           0
    dim_date     datekey         731          731           0

All primary keys are unique. No duplicates found.


In [9]:
# DateKey integrity
query_ri_date = """
SELECT COUNT(*) AS orphaned_datekeys
FROM fact_sales fs
LEFT JOIN dim_date dd ON fs.datekey = dd.datekey
WHERE dd.datekey IS NULL
"""

# PharmacyID integrity
query_ri_pharmacy = """
SELECT COUNT(*) AS orphaned_pharmacyids
FROM fact_sales fs
LEFT JOIN dim_pharmacy dp ON fs.pharmacyid = dp.pharmacyid
WHERE dp.pharmacyid IS NULL
"""

# ProductID integrity
query_ri_product = """
SELECT COUNT(*) AS orphaned_productids
FROM fact_sales fs
LEFT JOIN dim_product dpr ON fs.productid = dpr.productid
WHERE dpr.productid IS NULL
"""

ri_date     = run_query(query_ri_date).iloc[0, 0]
ri_pharmacy = run_query(query_ri_pharmacy).iloc[0, 0]
ri_product  = run_query(query_ri_product).iloc[0, 0]

print(f"Orphaned DateKeys    (no match in DimDate):     {ri_date}")
print(f"Orphaned PharmacyIDs (no match in DimPharmacy): {ri_pharmacy}")
print(f"Orphaned ProductIDs  (no match in DimProduct):  {ri_product}")
print()

if ri_date == 0 and ri_pharmacy == 0 and ri_product == 0:
    print("Referential integrity confirmed. All foreign keys have matching dimension records.")
else:
    print("WARNING: Orphaned foreign keys found. These rows will be lost in any JOIN.")

Orphaned DateKeys    (no match in DimDate):     0
Orphaned PharmacyIDs (no match in DimPharmacy): 0
Orphaned ProductIDs  (no match in DimProduct):  0

Referential integrity confirmed. All foreign keys have matching dimension records.


In [10]:
query_dates = """
SELECT
    MIN(date)               AS earliest_date,
    MAX(date)               AS latest_date,
    COUNT(DISTINCT year)    AS years_covered,
    COUNT(DISTINCT quarter) AS quarters_covered,
    COUNT(*)                AS total_days
FROM dim_date
"""

print("── DimDate coverage ──")
print(run_query(query_dates).to_string(index=False))


── DimDate coverage ──
earliest_date latest_date  years_covered  quarters_covered  total_days
   2024-01-01  2025-12-31              2                 4         731


In [11]:
# Confirm FactSales transactions fall within the date range
query_fs_dates = """
SELECT
    MIN(dd.date)                    AS earliest_transaction,
    MAX(dd.date)                    AS latest_transaction,
    COUNT(DISTINCT fs.datekey)      AS distinct_transaction_days
FROM fact_sales fs
JOIN dim_date dd ON fs.datekey = dd.datekey
"""

print("── FactSales transaction date range ──")
print(run_query(query_fs_dates).to_string(index=False))


── FactSales transaction date range ──
earliest_transaction latest_transaction  distinct_transaction_days
          2024-01-01         2025-12-31                        731


In [12]:
# Transactions per year — confirm both years are represented
query_yearly = """
SELECT
    dd.year,
    COUNT(*)                         AS transactions,
    ROUND(SUM(fs.revenueeur), 2)     AS total_revenue_eur
FROM fact_sales fs
JOIN dim_date dd ON fs.datekey = dd.datekey
GROUP BY dd.year
ORDER BY dd.year
"""

print("── Transactions and revenue by year ──")
print(run_query(query_yearly).to_string(index=False))

── Transactions and revenue by year ──
 year  transactions  total_revenue_eur
 2024         30312         4223414.13
 2025         31827         4410563.18


In [13]:
query_neg_margin = """
SELECT COUNT(*) AS negative_margin_rows
FROM fact_sales
WHERE margineur < 0
"""

neg = run_query(query_neg_margin).iloc[0, 0]
print(f"Transactions with negative margin: {neg}")
print("Expected: 0")

Transactions with negative margin: 0
Expected: 0


In [14]:
query_margin_check = """
SELECT COUNT(*) AS margin_mismatch_rows
FROM fact_sales
WHERE ABS(margineur - (revenueeur - costeur)) > 0.01
"""

mismatch = run_query(query_margin_check).iloc[0, 0]
print(f"Rows where MarginEUR != RevenueEUR - CostEUR: {mismatch}")
print("Expected: 0")

Rows where MarginEUR != RevenueEUR - CostEUR: 0
Expected: 0


In [15]:
query_zero_rev = """
SELECT COUNT(*) AS zero_or_negative_revenue
FROM fact_sales
WHERE revenueeur <= 0
"""

zero_rev = run_query(query_zero_rev).iloc[0, 0]
print(f"Transactions with zero or negative revenue: {zero_rev}")
print("Expected: 0")

Transactions with zero or negative revenue: 0
Expected: 0


In [16]:
query_discontinued = """
SELECT
    COUNT(*)                        AS transactions_with_discontinued,
    COUNT(DISTINCT fs.productid)    AS distinct_discontinued_products,
    ROUND(SUM(fs.revenueeur), 2)    AS revenue_from_discontinued
FROM fact_sales fs
JOIN dim_product dpr ON fs.productid = dpr.productid
WHERE dpr.isdiscontinued = 'Yes'
"""

print("── Sales from discontinued products ──")
print(run_query(query_discontinued).to_string(index=False))
print()
print("Note: Sales from discontinued products may represent old stock clearance.")
print("These will be included in analysis unless a decision is made to exclude them.")

── Sales from discontinued products ──
 transactions_with_discontinued  distinct_discontinued_products  revenue_from_discontinued
                           5045                              35                  696351.08

Note: Sales from discontinued products may represent old stock clearance.
These will be included in analysis unless a decision is made to exclude them.


In [17]:
query_daily = """
SELECT
    MIN(daily_count)               AS min_transactions_per_day,
    MAX(daily_count)               AS max_transactions_per_day,
    ROUND(AVG(daily_count), 1)     AS avg_transactions_per_day
FROM (
    SELECT datekey, COUNT(*) AS daily_count
    FROM fact_sales
    GROUP BY datekey
) daily
"""

print("── Daily transaction volume ──")
print(run_query(query_daily).to_string(index=False))

── Daily transaction volume ──
 min_transactions_per_day  max_transactions_per_day  avg_transactions_per_day
                       50                       123                      85.0


In [18]:
summary = """
DATA PROFILING SUMMARY
======================

Dataset: PharmaBI Retail Pharmacy Sales
Profiled: [fill in date]

ROW COUNTS
----------
fact_sales:   62,139  ✓
dim_date:        731  ✓
dim_pharmacy:    120  ✓
dim_product:     220  ✓

NULLS
-----
FactSales:    No nulls in any column. Clean.
DimPharmacy:  No nulls in any column. Clean.
DimProduct:   185 nulls in DiscontinuedDate — expected (185 active products have no discontinued date).
DimDate:      No nulls in any column. Clean.

DUPLICATES
----------
All primary keys are unique across all four tables. No duplicates.

REFERENTIAL INTEGRITY
---------------------
All foreign keys in FactSales have matching records in their dimension tables.
No orphaned transactions.

DATE RANGE
----------
DimDate covers 1 Jan 2024 to 31 Dec 2025 (731 days including leap day).
All FactSales transactions fall within this range.

BUSINESS LOGIC
--------------
- No negative margins found.
- MarginEUR = RevenueEUR - CostEUR confirmed for all rows (within 0.01 rounding tolerance).
- No zero or negative revenue transactions.
- [TBD] Discontinued product sales: fill in count after running Check 4.
- [TBD] Daily transaction range: fill in min/max/avg after running Check 5.

DECISIONS FOR ANALYSIS
----------------------
1. DiscontinuedDate nulls: expected — do not treat as a data quality issue.
2. StoreSizeBand values are S/M/L — map to Small/Medium/Large in analysis notebooks for display.
3. Discontinued products: [TBD — decide whether to include or exclude after reviewing Check 4 output].
4. PromoFlag is a Yes/No string — filter using WHERE promoflag = 'Yes' in analysis queries.
5. IsGeneric and IsDiscontinued are Yes/No strings — same pattern applies.

OVERALL STATUS
--------------
Data is clean. No blocking issues found. Safe to proceed to EDA.
"""

print(summary)


DATA PROFILING SUMMARY

Dataset: PharmaBI Retail Pharmacy Sales
Profiled: [fill in date]

ROW COUNTS
----------
fact_sales:   62,139  ✓
dim_date:        731  ✓
dim_pharmacy:    120  ✓
dim_product:     220  ✓

NULLS
-----
FactSales:    No nulls in any column. Clean.
DimPharmacy:  No nulls in any column. Clean.
DimProduct:   185 nulls in DiscontinuedDate — expected (185 active products have no discontinued date).
DimDate:      No nulls in any column. Clean.

DUPLICATES
----------
All primary keys are unique across all four tables. No duplicates.

REFERENTIAL INTEGRITY
---------------------
All foreign keys in FactSales have matching records in their dimension tables.
No orphaned transactions.

DATE RANGE
----------
DimDate covers 1 Jan 2024 to 31 Dec 2025 (731 days including leap day).
All FactSales transactions fall within this range.

BUSINESS LOGIC
--------------
- No negative margins found.
- MarginEUR = RevenueEUR - CostEUR confirmed for all rows (within 0.01 rounding tolerance).
-